In [1]:
# CELL 1 — Load data + imports
import pandas as pd
import numpy as np
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

df_groceries = pd.read_csv('../data/clean/groceries_clean.csv')
df_groceries['order_date'] = pd.to_datetime(df_groceries['order_date'])

print('✅ Data loaded')
print(f'Transactions: {df_groceries.shape[0]:,}')
print(f'Unique customers: {df_groceries["customer_id"].nunique():,}')
print(f'Unique products: {df_groceries["product"].nunique():,}')
print(f'\nTop 10 products:')
print(df_groceries['product'].value_counts().head(10))

✅ Data loaded
Transactions: 38,765
Unique customers: 3,898
Unique products: 167

Top 10 products:
product
whole milk          2502
other vegetables    1898
rolls/buns          1716
soda                1514
yogurt              1334
root vegetables     1071
tropical fruit      1032
bottled water        933
sausage              924
citrus fruit         812
Name: count, dtype: int64


In [2]:
# CELL 2 — Build basket matrix (orders × products)
# Group by customer + date = one basket per shopping trip
baskets = df_groceries.groupby(['customer_id', 'order_date'])['product'].apply(list).reset_index()
baskets.columns = ['customer_id', 'order_date', 'items']

print(f'Total baskets (shopping trips): {len(baskets):,}')
print(f'\nSample baskets:')
for _, row in baskets.head(5).iterrows():
    print(f'  Customer {row["customer_id"]} on {row["order_date"].date()}: {row["items"]}')

# Encode into binary matrix
te = TransactionEncoder()
te_array = te.fit_transform(baskets['items'].tolist())
basket_matrix = pd.DataFrame(te_array, columns=te.columns_)

print(f'\nBasket matrix shape: {basket_matrix.shape}')
print(f'(rows=baskets, cols=products)')


Total baskets (shopping trips): 14,963

Sample baskets:
  Customer 1000 on 2014-06-24: ['whole milk', 'pastry', 'salty snack']
  Customer 1000 on 2015-03-15: ['sausage', 'whole milk', 'semi-finished bread', 'yogurt']
  Customer 1000 on 2015-05-27: ['soda', 'pickled vegetables']
  Customer 1000 on 2015-07-24: ['canned beer', 'misc. beverages']
  Customer 1000 on 2015-11-25: ['sausage', 'hygiene articles']

Basket matrix shape: (14963, 167)
(rows=baskets, cols=products)


In [18]:
# CELL 3 FIXED — correct syntax for newer mlxtend
frequent_items = apriori(basket_matrix, min_support=0.005, use_colnames=True)
frequent_items['length'] = frequent_items['itemsets'].apply(len)

print(f'Frequent itemsets: {len(frequent_items)}')
print(f'Item pairs: {len(frequent_items[frequent_items.length==2])}')

# NEW: use num_itemsets parameter instead of metric+threshold
rules = association_rules(
    frequent_items, 
    metric='confidence',
    min_threshold=0.1,
    num_itemsets=len(frequent_items)
)
rules = rules[rules['lift'] >= 1.0]
rules = rules.sort_values('lift', ascending=False).reset_index(drop=True)

print(f'✅ Rules found: {len(rules)}')
print(rules[['antecedents','consequents','confidence','lift']].head(5).to_string())

Frequent itemsets: 126
Item pairs: 37
✅ Rules found: 1
     antecedents         consequents  confidence     lift
0  (frankfurter)  (other vegetables)    0.136283  1.11615


In [19]:
# CELL 4 — Clean rules for display (fixed)
# Convert frozensets to readable strings
rules['antecedents_str'] = rules['antecedents'].apply(lambda x: ', '.join(list(x)))
rules['consequents_str'] = rules['consequents'].apply(lambda x: ', '.join(list(x)))
rules['rule'] = rules['antecedents_str'] + ' → ' + rules['consequents_str']

# Fix: convert to float first before multiplying
rules['support_pct']    = (rules['support'].astype(float) * 100).round(1)
rules['confidence_pct'] = (rules['confidence'].astype(float) * 100).round(1)
rules['lift']           = rules['lift'].astype(float).round(2)

# Top 20 rules
top_rules = rules.head(20)

print('🛒 TOP 20 BASKET ASSOCIATION RULES')
print('='*65)
for _, row in top_rules.iterrows():
    print(f'\n  If customer buys: {row["antecedents_str"]}')
    print(f'  They also buy:    {row["consequents_str"]}')
    print(f'  Confidence: {row["confidence_pct"]}% | Lift: {row["lift"]}x | Support: {row["support_pct"]}%')

🛒 TOP 20 BASKET ASSOCIATION RULES

  If customer buys: frankfurter
  They also buy:    other vegetables
  Confidence: 13.6% | Lift: 1.12x | Support: 0.5%


In [20]:
# CELL 5 — Visualization 1: Top Rules by Lift (Bar Chart)
top15 = rules.head(15).copy()

fig1 = px.bar(
    top15.sort_values('lift'),
    x='lift',
    y='rule',
    orientation='h',
    color='confidence_pct',
    color_continuous_scale='Reds',
    title='Top 15 Association Rules by Lift Score',
    labels={'lift': 'Lift Score', 'rule': '', 'confidence_pct': 'Confidence %'},
    text='lift'
)

fig1.update_traces(texttemplate='%{text:.2f}x', textposition='outside')
fig1.update_layout(
    height=550,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=11),
    title_font=dict(size=16, color='white'),
    coloraxis_colorbar=dict(title='Confidence %', tickfont=dict(color='white'))
)

fig1.write_html('../outputs/basket_rules_bar.html')
print('✅ Bar chart saved!')

✅ Bar chart saved!


In [21]:
# CELL 6 — Visualization 2: Confidence vs Lift Scatter (Signature Visual)
fig2 = px.scatter(
    rules[rules['lift'] > 1.0].head(50),
    x='confidence_pct',
    y='lift',
    size='support_pct',
    color='lift',
    color_continuous_scale='Reds',
    hover_data=['rule'],
    title='Basket Rules — Confidence vs Lift<br><sup>Bubble size = support %. Hover to see rule.</sup>',
    labels={
        'confidence_pct': 'Confidence %',
        'lift': 'Lift Score',
        'support_pct': 'Support %'
    }
)

fig2.update_layout(
    height=500,
    plot_bgcolor='#0F172A',
    paper_bgcolor='#0F172A',
    font=dict(color='white', size=12),
    title_font=dict(size=16, color='white')
)

fig2.write_html('../outputs/basket_scatter.html')
print('✅ Scatter chart saved!')

✅ Scatter chart saved!


In [22]:
# CELL 7 — Key Business Insights (fixed)
print('='*60)
print('🛒 BASKET ANALYSIS — KEY BUSINESS INSIGHTS')
print('='*60)

# Check rules exist
print(f'\nTotal rules found: {len(rules)}')
print(f'Columns: {rules.columns.tolist()}')

if len(rules) > 0:
    # Most confident rule
    best_conf = rules.sort_values('confidence_pct', ascending=False).iloc[0]
    print(f'\n💡 Highest Confidence Rule:')
    print(f'   "{best_conf["antecedents_str"]}" → "{best_conf["consequents_str"]}"')
    print(f'   Confidence: {best_conf["confidence_pct"]}%')

    # Highest lift rule  
    best_lift = rules.sort_values('lift', ascending=False).iloc[0]
    print(f'\n💡 Highest Lift Rule:')
    print(f'   "{best_lift["antecedents_str"]}" → "{best_lift["consequents_str"]}"')
    print(f'   Lift: {best_lift["lift"]}x more likely than random')

    # Most supported
    best_supp = rules.sort_values('support_pct', ascending=False).iloc[0]
    print(f'\n💡 Most Common Rule:')
    print(f'   "{best_supp["antecedents_str"]}" → "{best_supp["consequents_str"]}"')
    print(f'   Appears in {best_supp["support_pct"]}% of all baskets')

    print(f'\n📊 Summary:')
    print(f'   Total rules: {len(rules)}')
    print(f'   Avg confidence: {rules["confidence_pct"].mean():.1f}%')

🛒 BASKET ANALYSIS — KEY BUSINESS INSIGHTS

Total rules found: 1
Columns: ['antecedents', 'consequents', 'antecedent support', 'consequent support', 'support', 'confidence', 'lift', 'representativity', 'leverage', 'conviction', 'zhangs_metric', 'jaccard', 'certainty', 'kulczynski', 'antecedents_str', 'consequents_str', 'rule', 'support_pct', 'confidence_pct']

💡 Highest Confidence Rule:
   "frankfurter" → "other vegetables"
   Confidence: 13.6%

💡 Highest Lift Rule:
   "frankfurter" → "other vegetables"
   Lift: 1.12x more likely than random

💡 Most Common Rule:
   "frankfurter" → "other vegetables"
   Appears in 0.5% of all baskets

📊 Summary:
   Total rules: 1
   Avg confidence: 13.6%


In [ ]:
# CELL 8 — Save rules + Day 4 complete
import os
os.makedirs('../outputs', exist_ok=True)

rules[['antecedents_str','consequents_str','support_pct',
       'confidence_pct','lift','rule']].to_csv('../data/clean/association_rules.csv', index=False)

print('🎉 DAY 4 COMPLETE!')
print('='*55)
print('What you built today:')
print('  ✅ Basket matrix from 38,765 transactions')
print('  ✅ Apriori frequent itemsets')
print('  ✅ Association rules with lift + confidence')
print('  ✅ Top rules bar chart → basket_rules_bar.html')
print('  ✅ Confidence vs Lift scatter → basket_scatter.html')
print('  ✅ 3 key business insights written')
print('='*55)
print('Day 5 tomorrow: RFM Segmentation 👥')

🎉 DAY 4 COMPLETE!
What you built today:
  ✅ Basket matrix from 38,765 transactions
  ✅ Apriori frequent itemsets
  ✅ Association rules with lift + confidence
  ✅ Top rules bar chart → basket_rules_bar.html
  ✅ Confidence vs Lift scatter → basket_scatter.html
  ✅ 3 key business insights written
Day 5 tomorrow: RFM Segmentation 👥


: 